# Imports

In [1]:
import pandas as pd
import numpy as np
import torch
from torch import nn
from torch.utils.data import Dataset
from torch.utils.data import DataLoader
from sklearn.model_selection import train_test_split
from torchinfo import summary
from tqdm import tqdm
from sklearn.metrics import r2_score

# Constants

In [2]:
device = 'cuda'

# Data Processing

In [3]:
df = pd.read_csv('../AAPL_dataset.csv')
df = df.drop(columns=['Date', 'Open', 'High', 'Low', 'Adj Close', 'Volume'])
print(df.head())

      Close
0  0.128348
1  0.121652
2  0.112723
3  0.115513
4  0.118862


In [4]:
data_np = df.to_numpy().astype(np.float32)
split_idx = int(len(data_np) * 0.7)
train_np = data_np[:split_idx]
test_np = data_np[split_idx:]

def create_windows(data, win_len):
    data = data.squeeze()
    n = len(data)

    windows_x = []
    windows_y = []

    for i in range(n - win_len):
        window = data[i: i + win_len]

        log_window = np.log(window)
        diff_window = np.diff(log_window).reshape(-1, 1)   # shape: (win_len - 1,)

        target = (np.log(data[i + win_len]) - log_window[-1]).reshape(1)

        windows_x.append(diff_window)
        windows_y.append(target)

    return np.array(windows_x), np.array(windows_y)

win_len = 100
train_x, train_y = create_windows(train_np, win_len)
print(train_x.shape)
print(train_y.shape)
test_x, test_y = create_windows(test_np, win_len)

train_len = len(train_np)
test_len = len(test_np)

(7636, 99, 1)
(7636, 1)


In [5]:
class TupleDataset(Dataset):
    def __init__(self, x, y):
        self.x = torch.from_numpy(x)
        self.y = torch.from_numpy(y)

    def __len__(self):
        return len(self.x)

    def __getitem__(self, idx):
        return self.x[idx], self.y[idx]
    
    def shape(self):
        return self.x.shape

In [6]:
train_dataset = TupleDataset(train_x, train_y)
test_dataset = TupleDataset(test_x, test_y)

train_loader = DataLoader(
    train_dataset,
    batch_size=32,
    shuffle=False
)

test_loader = DataLoader(
    test_dataset,
    batch_size=32,
    shuffle=False
)

In [7]:
print(train_dataset.shape())

torch.Size([7636, 99, 1])


# Models

BiGRU

In [8]:
class BiGRU(nn.Module):
    def __init__(self, input_features):
        super().__init__()
        
        self.bigru = nn.GRU(
            input_features, 64,
            num_layers=3,
            bidirectional=True,
            batch_first=True,
            dropout=0.2
        )
        
        self.fc = nn.Sequential(
            nn.Linear(128, 32),
            nn.ReLU(),

            nn.Linear(32, 16),
            nn.ReLU(),
            nn.Dropout(0.2),

            nn.Linear(16, 1)
        )
        
    def forward(self, x):
        _, h = self.bigru(x)
        
        # h shape: (num_layers * num_directions, batch, hidden)
        num_layers = self.bigru.num_layers
        num_directions = 2 if self.bigru.bidirectional else 1
        
        h = h.view(num_layers, num_directions, x.size(0), 64)
        h = h[-1]  # last layer → (2, batch, 64)
        
        h = torch.cat((h[0], h[1]), dim=1)  # (batch, 128)
        
        return self.fc(h)

In [9]:
bigru_model = BiGRU(input_features=1)
summary(
    bigru_model,
    train_dataset.shape()
)

Layer (type:depth-idx)                   Output Shape              Param #
BiGRU                                    [7636, 1]                 --
├─GRU: 1-1                               [7636, 99, 128]           174,720
├─Sequential: 1-2                        [7636, 1]                 --
│    └─Linear: 2-1                       [7636, 32]                4,128
│    └─ReLU: 2-2                         [7636, 32]                --
│    └─Linear: 2-3                       [7636, 16]                528
│    └─ReLU: 2-4                         [7636, 16]                --
│    └─Dropout: 2-5                      [7636, 16]                --
│    └─Linear: 2-6                       [7636, 1]                 17
Total params: 179,393
Trainable params: 179,393
Non-trainable params: 0
Total mult-adds (G): 132.12
Input size (MB): 3.02
Forward/backward pass size (MB): 777.10
Params size (MB): 0.72
Estimated Total Size (MB): 780.84

In [10]:
class BiLSTM(nn.Module):
    def __init__(self, input_features):
        super().__init__()
        
        self.bilstm = nn.LSTM(
            input_features, 64,
            num_layers=3,
            bidirectional=True,
            batch_first=True,
            dropout=0.2
        )
        
        self.fc = nn.Sequential(
            nn.Linear(128, 32),
            nn.ReLU(),

            nn.Linear(32, 16),
            nn.ReLU(),
            nn.Dropout(0.2),

            nn.Linear(16, 1)
        )
        
    def forward(self, x):
        _, (h, _) = self.bilstm(x)
        
        # shape: (num_layers * num_directions, batch, hidden)
        h = h.view(3, 2, x.size(0), 64)  # (layers, directions, batch, hidden)
        h = h[-1]                        # last layer -> (2, batch, 64)
        
        h = torch.cat((h[0], h[1]), dim=1)  # (batch, 128)
        
        return self.fc(h)

In [11]:
bilstm_model = BiLSTM(input_features=1)
summary(
    bilstm_model,
    train_dataset.shape()
)

Layer (type:depth-idx)                   Output Shape              Param #
BiLSTM                                   [7636, 1]                 --
├─LSTM: 1-1                              [7636, 99, 128]           232,960
├─Sequential: 1-2                        [7636, 1]                 --
│    └─Linear: 2-1                       [7636, 32]                4,128
│    └─ReLU: 2-2                         [7636, 32]                --
│    └─Linear: 2-3                       [7636, 16]                528
│    └─ReLU: 2-4                         [7636, 16]                --
│    └─Dropout: 2-5                      [7636, 16]                --
│    └─Linear: 2-6                       [7636, 1]                 17
Total params: 237,633
Trainable params: 237,633
Non-trainable params: 0
Total mult-adds (G): 176.15
Input size (MB): 3.02
Forward/backward pass size (MB): 777.10
Params size (MB): 0.95
Estimated Total Size (MB): 781.07

FC

In [12]:
class FC(nn.Module):
    def __init__(self, input_features):
        super().__init__()
        
        self.fc1 = nn.Sequential(
            nn.Linear(input_features, 16),
            nn.ReLU(),

            nn.Linear(16, 32),
            nn.ReLU(),

            nn.Linear(32, 128)
        )
        
        self.fc = nn.Sequential(
            nn.Linear(128, 32),
            nn.ReLU(),

            nn.Linear(32, 16),
            nn.ReLU(),
            nn.Dropout(0.2),

            nn.Linear(16, 1)
        )
        
    def forward(self, x):
        x = x.squeeze(-1)
        out = self.fc1(x)
        
        return self.fc(out)

In [13]:
fc_model = FC(input_features=99)
summary(
    fc_model,
    train_dataset.shape()
)

Layer (type:depth-idx)                   Output Shape              Param #
FC                                       [7636, 1]                 --
├─Sequential: 1-1                        [7636, 128]               --
│    └─Linear: 2-1                       [7636, 16]                1,600
│    └─ReLU: 2-2                         [7636, 16]                --
│    └─Linear: 2-3                       [7636, 32]                544
│    └─ReLU: 2-4                         [7636, 32]                --
│    └─Linear: 2-5                       [7636, 128]               4,224
├─Sequential: 1-2                        [7636, 1]                 --
│    └─Linear: 2-6                       [7636, 32]                4,128
│    └─ReLU: 2-7                         [7636, 32]                --
│    └─Linear: 2-8                       [7636, 16]                528
│    └─ReLU: 2-9                         [7636, 16]                --
│    └─Dropout: 2-10                     [7636, 16]                --
│   

# Training

BiGRU

In [14]:
# criterion = nn.HuberLoss(delta=1.0)
# optimizer = torch.optim.Adam(bigru_model.parameters(), lr=1e-5)
# num_epochs = 100

# for epoch in range(num_epochs):
#     bigru_model.train()
#     train_loss = 0
#     pbar = tqdm(train_loader, desc=f"Epoch {epoch+1}/{num_epochs}")
    
#     for inputs, targets in pbar:
#         inputs = inputs.to(device)
#         targets = targets.to(device)

#         optimizer.zero_grad()
#         outputs = bigru_model(inputs)
        
#         loss = criterion(outputs, targets)
#         loss.backward()
#         optimizer.step()
        
#         train_loss += loss.item()
        
#     print(f"Epoch {epoch+1}: Train Loss = {train_loss / len(train_loader):.6f}")

In [15]:
# torch.save(bigru_model.state_dict(), "bigru_model.pth")

BiLSTM

In [16]:
# criterion = nn.HuberLoss(delta=1.0)
# optimizer = torch.optim.Adam(bilstm_model.parameters(), lr=1e-5)
# num_epochs = 100

# for epoch in range(num_epochs):
#     bilstm_model.train()
#     train_loss = 0
#     pbar = tqdm(train_loader, desc=f"Epoch {epoch+1}/{num_epochs}")
    
#     for inputs, targets in pbar:
#         inputs = inputs.to(device)
#         targets = targets.to(device)

#         optimizer.zero_grad()
        
#         outputs = bilstm_model(inputs)
#         loss = criterion(outputs, targets)
#         loss.backward()
#         optimizer.step()
        
#         train_loss += loss.item()
        
#     print(f"Epoch {epoch+1}: Train Loss = {train_loss / len(train_loader):.6f}")

In [17]:
# torch.save(bilstm_model.state_dict(), "bilstm_model.pth")

FC

In [18]:
criterion = nn.HuberLoss(delta=1.0)
optimizer = torch.optim.Adam(fc_model.parameters(), lr=1e-5)
num_epochs = 100

for epoch in range(num_epochs):
    fc_model.train()
    train_loss = 0
    pbar = tqdm(train_loader, desc=f"Epoch {epoch+1}/{num_epochs}")
    
    for inputs, targets in pbar:
        inputs = inputs.to(device)
        targets = targets.to(device)

        optimizer.zero_grad()
        
        outputs = fc_model(inputs)
        loss = criterion(outputs, targets)
        loss.backward()
        optimizer.step()
        
        train_loss += loss.item()
        
    print(f"Epoch {epoch+1}: Train Loss = {train_loss / len(train_loader):.6f}")

Epoch 1/100: 100%|██████████| 239/239 [00:01<00:00, 202.21it/s]


Epoch 1: Train Loss = 0.000793


Epoch 2/100: 100%|██████████| 239/239 [00:00<00:00, 243.41it/s]


Epoch 2: Train Loss = 0.000692


Epoch 3/100: 100%|██████████| 239/239 [00:01<00:00, 237.95it/s]


...

Epoch 98: Train Loss = 0.000503


Epoch 99/100: 100%|██████████| 239/239 [00:00<00:00, 262.76it/s]


Epoch 99: Train Loss = 0.000507


Epoch 100/100: 100%|██████████| 239/239 [00:00<00:00, 319.20it/s]

Epoch 100: Train Loss = 0.000504


In [19]:
torch.save(fc_model.state_dict(), "fc_model.pth")

# Save Dataset

In [20]:
torch.save(train_dataset, "train_dataset.pt")
torch.save(test_dataset, "test_dataset.pt")